# 15. Latent Space Fusion — demographic + fused multi-modal feature-sets (FOC-178, phase F4)

Phase F4's last notebook. Two jobs:

1. **Demographic embeddings** (`demo-features` arm) — each customer's `dim_customer` profile
   rendered as a short deterministic text and embedded with the SAME MiniLM-L6 backbone the
   text arm uses (the nb14 winner), so modality differences come from content, not encoder.
2. **Fusion** (`latent-fusion` arm) — the joint latent space: face (512) + text (384) +
   demographic (384) blocks, each L2-normalized, **concatenated statelessly** — plus a
   `latent-pure` ABLATION (the 1280 embedding dims WITHOUT the tabular base) for modality
   attribution.

Evaluation is the unified runner protocol on ALL THREE axes; **`random-grouped`
(cohort-random, customer-grouped) is the PRIMARY header axis** (977 test rows, 13 positives,
chance 0.0133 in the F3 headline). Every table prints test positives and chance level.

Honest expectation (plan §7 pre-registration): at 5,302 transactions / ~90 frauds these
embeddings almost certainly will NOT beat chance on the customer-grouped axes — demographic
and text blocks re-encode signal the tabular arms already see, and the face block is a
customer-identity proxy at best. **Null results are findings, not failures.**

In [1]:
# Runtime provenance: this notebook must be reproducible from the phase venv alone.
import platform
import sys

print('python:', sys.version.split()[0], '| platform:', platform.platform())
import numpy as np

print('numpy:', np.__version__)
try:
    import torch

    print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
except ImportError:
    print('torch: NOT importable — embedding arms will surface SKIPPED rows')
try:
    import sentence_transformers

    print('sentence-transformers:', sentence_transformers.__version__)
except ImportError:
    print('sentence-transformers: not importable')
import arms_demo
import arms_fusion

print('demo npz artifact:', arms_demo.DEMO_EMBEDDINGS_NPZ_PATH,
      '(exists: %s)' % arms_demo.DEMO_EMBEDDINGS_NPZ_PATH.is_file())
print('fusion modality blocks: face %d + text %d + demo %d = %d dims' % (
    len(arms_fusion.FACE_FEATURES), len(arms_fusion.TEXT_FEATURES),
    len(arms_fusion.DEMO_FEATURES), len(arms_fusion.FUSED_FEATURES)))

python: 3.11.9 | platform: Windows-10-10.0.26200-SP0
numpy: 2.4.6


torch: 2.11.0+cu128 | cuda: True


sentence-transformers: 6.0.1
demo npz artifact: C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-178-f4\data\demo_embeddings.npz (exists: True)
fusion modality blocks: face 512 + text 384 + demo 384 = 1280 dims


In [2]:
import pandas as pd

import arms_text
from fraud_pipeline import (
    AXES,
    ArmSkipped,
    DEFAULT_RESULTS_PATH,
    _features_xgb_client,
    load_enriched,
    load_results,
    print_comparison_table,
    register_arm,
    run_arm_on_axis,
    xgb_params,
)
from xgboost import XGBClassifier

# Canonical enriched frame + labels from the unified runner (the nb7/nb8 data
# section, shared by every arm — loaded once, used by everything below).
enriched, y = load_enriched()
print(
    'fraud txns: %d of %d (%.2f%%) across %d unique customers'
    % (int(y.sum()), len(y), 100 * y.mean(), enriched['customer'].nunique())
)
missing = arms_fusion.check_dependencies()
print('dependency probe:', missing if missing else 'face npz + text/demo encoders OK (local caches)')

fraud txns: 91 of 5302 (1.72%) across 100 unique customers
dependency probe: face npz + text/demo encoders OK (local caches)


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-178-f4\src\funs.py:197: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  trxns_data["timestamp"] = pd.to_datetime(


## Fusion choice: concat of L2-normalized blocks — not a fitted projection, not alignment

Stated BEFORE any results, because it is a design commitment, not a tuned outcome:

- **Any FITTED projection (PCA, CCA, Procrustes, a learned fusion MLP) must be fit somewhere.**
  The runner contract evaluates `build_features` on the FULL enriched frame BEFORE the split —
  anything fit there sees test structure, i.e. leaks. Refitting inside `make_model` per split
  would fix the leakage but make features split-dependent, breaking the cached label-free
  extraction contract every other F4 arm follows.
- **Alignment needs paired anchors and data we do not have** — 100 customers cannot support a
  credible shared space, and there is no cross-domain anchor set.
- **Concat of unit-norm blocks IS the leak-free "one latent space"**: each modality contributes
  exactly unit length (no scale dominance), cross-modal geometry is directly readable, and the
  downstream XGB is per-feature monotone-invariant so block scaling would not change its trees —
  the L2 normalization matters for the GEOMETRY consumers (distance/angle thresholds) that the
  F5 threshold layer is planned to need.

Caveats carried from nb13/nb14 (full versions there): the face block is a **customer-identity
proxy with a pre-registered NO-signal expectation** and an ethical caveat against any
appearance-based scoring; the text block is **synthesized** from existing tabular fields, so it
largely re-encodes known signal; the demographic block rides the same backbone for the same
reason. Fusion can therefore only re-arrange signal the tabular arms already have — the
question is whether the joint GEOMETRY adds anything the one-hot encoding cannot.

In [3]:
# --- demographic profiles: what gets embedded -------------------------------
# The 8 dim_customer fields (constant per customer, asserted) render into one
# short text per customer; the corpus is content-addressed (digest) so an
# edited dim_customer re-encodes instead of silently returning stale vectors.
customer_ids, profile_texts = arms_demo.customer_profiles(enriched)
print('customers: %d | profile columns: %s' % (
    len(customer_ids), ', '.join(arms_demo.PROFILE_COLUMNS)))
for cid, text in list(zip(customer_ids, profile_texts))[:3]:
    print('  %s -> "%s"' % (cid, text))

demo_frame = arms_demo.append_features(enriched)  # encodes once, writes the npz
print(
    'demo features: %d cols over %d customers | npz artifact: %s'
    % (len(arms_fusion.DEMO_FEATURES), len(customer_ids),
       arms_demo.DEMO_EMBEDDINGS_NPZ_PATH.is_file())
)

customers: 100 | profile columns: gender, age, employment_industry, employment_status, account_tenure_years, income_band, device, channel
  C12976926337644 -> "Customer profile: gender F, age 39, unemployed in the technology industry, income band 2_20k_40k, account tenure 9 years, usually on mobile, branch channel."
  C14368611669296 -> "Customer profile: gender M, age 49, employed in the education industry, income band 3_40k_60k, account tenure 10 years, usually on desktop, mobile_app channel."
  C17694553858863 -> "Customer profile: gender F, age 34, employed in the technology industry, income band 2_20k_40k, account tenure 16 years, usually on desktop, mobile_app channel."
demo features: 384 cols over 100 customers | npz artifact: True


In [4]:
# --- arm registrations --------------------------------------------------------
# demo-features and latent-fusion are REGISTERED BY THE RUNNER at import time
# (fraud_pipeline.py F4 integration — the same lazy-wrapper pattern this cell
# used before the integration landed); re-registering here would raise
# 'arm already registered'. The notebook evaluates them through the registry,
# so the runner rows and this notebook's rows share one code path by
# construction. Only the latent-pure ABLATION is notebook-local.
from fraud_pipeline import ARMS

assert 'demo-features' in ARMS and 'latent-fusion' in ARMS, (
    'runner-level F4 arms missing — fraud_pipeline.py registration not loaded'
)


def _build_latent_pure(enr):
    # ABLATION, not a headline arm: the 1280 embedding dims WITHOUT the
    # tabular base — how much of the (null) fused result is embeddings alone
    # vs the xgb-client matrix riding along.
    missing = arms_fusion.check_dependencies()
    if missing is not None:
        raise ArmSkipped('latent-pure: missing dependency (%s)' % missing)
    fused = arms_fusion.append_features(enr)
    return fused[arms_fusion.FUSED_FEATURES].copy()


def _make_latent_pure(y_fit):
    missing = arms_fusion.check_dependencies()
    if missing is not None:
        raise ArmSkipped('latent-pure: missing dependency (%s)' % missing)
    from xgboost import XGBClassifier

    return XGBClassifier(**xgb_params(y_fit))


register_arm(
    'latent-pure',
    'ABLATION on the fused space: the 1280 embedding dims alone (no tabular '
    'base) — modality attribution, not a headline arm',
    make_model=_make_latent_pure,
    build_features=_build_latent_pure,
    supports_cv=True,
)
print(
    'runner-level arms ready:', ', '.join(
        a for a in ('demo-features', 'latent-fusion') if a in ARMS
    ),
    '| notebook-local: latent-pure',
)

runner-level arms ready: demo-features, latent-fusion | notebook-local: latent-pure


In [5]:
# --- demo-features: all three axes ------------------------------------------
PRIMARY_AXIS = 'random-grouped'
AXIS_ORDER = ('random-grouped', 'grouped', 'chronological')

rows_demo = [run_arm_on_axis('demo-features', axis, enriched, y, cv=False) for axis in AXIS_ORDER]
print_comparison_table(rows_demo, title='demo-features — test metrics per axis (frozen threshold)')
for row in rows_demo:
    print(
        '%-16s test positives %-3d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
        % (row['axis'], row['test_positives'], row['pr_auc'], row['pr_auc_ci_low'],
           row['pr_auc_ci_high'], row['chance_level'], row['pr_auc'] - row['chance_level'],
           'covers chance' if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
           else 'separates'))


== demo-features — test metrics per axis (frozen threshold) ==
          axis           arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  n_features
random-grouped demo-features     ok              13        977        0.0133  0.0114         0.0064          0.0239   0.3714          0.2121           0.5345 0.0000               0.0000            0.7103         500
       grouped demo-features     ok              11        831        0.0132  0.0118         0.0059          0.0268   0.3518          0.1595           0.5668 0.0000               0.0000            0.8926         500
 chronological demo-features     ok              24       1061        0.0226  0.2341         0.0898          0.4184   0.6851          0.5077           0.8471 0.0769               0.1667            0.7709         500
random-grouped   test positives 13  | PR-AUC 0.0114 [CI 0.0064, 0.0239] 

In [6]:
# --- latent-fusion: all three axes ------------------------------------------
rows_fusion = [run_arm_on_axis('latent-fusion', axis, enriched, y, cv=False) for axis in AXIS_ORDER]
print_comparison_table(rows_fusion, title='latent-fusion — test metrics per axis (frozen threshold)')
for row in rows_fusion:
    print(
        '%-16s test positives %-3d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
        % (row['axis'], row['test_positives'], row['pr_auc'], row['pr_auc_ci_low'],
           row['pr_auc_ci_high'], row['chance_level'], row['pr_auc'] - row['chance_level'],
           'covers chance' if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
           else 'separates'))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


== latent-fusion — test metrics per axis (frozen threshold) ==
          axis           arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  n_features
random-grouped latent-fusion     ok              13        977        0.0133  0.0174         0.0084          0.0503   0.5348          0.3846           0.6793 0.0000               0.0000            0.6086        1396
       grouped latent-fusion     ok              11        831        0.0132  0.0194         0.0097          0.0480   0.6157          0.4687           0.7453 0.0000               0.0000            0.6046        1396
 chronological latent-fusion     ok              24       1061        0.0226  0.1260         0.0383          0.2739   0.7191          0.6280           0.8092 0.1333               0.0417            0.5263        1396
random-grouped   test positives 13  | PR-AUC 0.0174 [CI 0.0084, 0.0503] 

In [7]:
# --- latent-pure (ablation): PRIMARY axis only ------------------------------
row_pure = run_arm_on_axis('latent-pure', PRIMARY_AXIS, enriched, y, cv=False)
print_comparison_table(
    [row_pure], title='latent-pure (ablation) — %s only (frozen threshold)' % PRIMARY_AXIS
)
print(
    '%-16s test positives %-3d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
    % (row_pure['axis'], row_pure['test_positives'], row_pure['pr_auc'],
       row_pure['pr_auc_ci_low'], row_pure['pr_auc_ci_high'], row_pure['chance_level'],
       row_pure['pr_auc'] - row_pure['chance_level'],
       'covers chance'
       if row_pure['pr_auc_ci_low'] <= row_pure['chance_level'] <= row_pure['pr_auc_ci_high']
       else 'separates'))


== latent-pure (ablation) — random-grouped only (frozen threshold) ==
          axis         arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high  f1  recall_at_precision  frozen_threshold  n_features
random-grouped latent-pure     ok              13        977        0.0133  0.0173         0.0083          0.0475    0.541          0.3917           0.6824 0.0                  0.0            0.1827        1280
random-grouped   test positives 13  | PR-AUC 0.0173 [CI 0.0083, 0.0475] vs chance 0.0133 (delta +0.0040) -> covers chance


In [8]:
# --- F4 scoreboard: canonical runner rows (JSONL) + this notebook's runs -----
def latest_ok(axis, arm):
    # Latest ok row per (axis, arm) from the accumulated JSONL.
    matches = [
        r for r in load_results(DEFAULT_RESULTS_PATH)
        if r.get('axis') == axis and r.get('arm') == arm and r.get('status') == 'ok'
    ]
    return matches[-1] if matches else None


in_notebook = {(r['axis'], r['arm']): r for r in rows_demo + rows_fusion + [row_pure]}
score_arms = [
    'xgb-client', 'face-features', 'text-features-minilm-l6',
    'demo-features', 'latent-fusion', 'latent-pure',
]
score_rows = []
for axis in AXIS_ORDER:
    for arm in score_arms:
        row = latest_ok(axis, arm)
        if row is None and arm in ('demo-features', 'latent-fusion', 'latent-pure'):
            row = in_notebook.get((axis, arm))
            if row is not None:
                print('note (fallback to in-notebook row, cv=False): %s / %s' % (axis, arm))
        if row is None:
            print('absent from runner JSONL: %s / %s' % (axis, arm))
        else:
            score_rows.append(row)

print_comparison_table(
    score_rows,
    title='F4 latent-space scoreboard — embedding arms vs xgb-client '
          '(PRIMARY first; test positives + chance printed per axis below)',
)
for axis in AXIS_ORDER:
    subset = [r for r in score_rows if r['axis'] == axis]
    if subset:
        print(
            '%s: test positives %d | chance %.4f'
            % (axis, subset[0]['test_positives'], subset[0]['chance_level'])
        )
for row in score_rows:
    if row['axis'] == PRIMARY_AXIS:
        print(
            '%-26s PR-AUC %.4f vs chance %.4f (delta %+.4f) -> %s'
            % (row['arm'], row['pr_auc'], row['chance_level'],
               row['pr_auc'] - row['chance_level'],
               'covers chance'
               if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
               else 'separates')
        )

note (fallback to in-notebook row, cv=False): random-grouped / latent-pure
absent from runner JSONL: grouped / latent-pure
absent from runner JSONL: chronological / latent-pure

== F4 latent-space scoreboard — embedding arms vs xgb-client (PRIMARY first; test positives + chance printed per axis below) ==
          axis                     arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  cv_pr_auc_mean  cv_pr_auc_std  n_features
random-grouped              xgb-client     ok              13        977        0.0133  0.0144         0.0079          0.0300   0.4844          0.3334           0.6425 0.0000               0.0000            0.8776          0.6812         0.0768         116
random-grouped           face-features     ok              13        977        0.0133  0.0110         0.0070          0.0198   0.3989          0.2750           0.5332 0.0000      

In [9]:
# --- determinism: two fusion builds, aligned on ROW IDENTITY -----------------
# The F3 round-2 lesson: NEVER diff feature frames positionally. Both passes
# below run on the same enriched frame; pass 2 bypasses every cache
# (fresh FaceNet + fresh encodes), then both frames are asserted index-equal
# BEFORE the diff. A positional diff here would be meaningless by construction.
fused_a = arms_fusion.append_features(enriched)
fused_b = arms_fusion.append_features(enriched, use_cache=False)
assert fused_a.index.equals(fused_b.index), 'frame indices differ — cannot align'

max_delta = float(
    np.abs(
        fused_b[arms_fusion.FUSED_FEATURES].to_numpy()
        - fused_a[arms_fusion.FUSED_FEATURES].to_numpy()
    ).max()
)
print('index-aligned fusion rebuild: max |delta| = %.3e over %d x %d features'
      % (max_delta, len(enriched), len(arms_fusion.FUSED_FEATURES)))
assert max_delta < 1e-6, 'fusion build is not deterministic'

# The committed demo artifact must round-trip: disk reload == in-memory values.
reloaded = arms_demo.load_embeddings() if hasattr(arms_demo, 'load_embeddings') else None
if reloaded is None:  # the demo npz has its own reader (embed_customers path)
    npz = np.load(arms_demo.DEMO_EMBEDDINGS_NPZ_PATH, allow_pickle=False)
    disk = pd.DataFrame(npz['embeddings'].astype(np.float32), index=list(npz['customer_id']),
                        columns=arms_demo.demo_feature_columns(npz['embeddings'].shape[1]))
    reloaded = disk
disk_delta = float(
    np.abs(
        reloaded.sort_index().to_numpy()
        - demo_frame[arms_fusion.DEMO_FEATURES].groupby(enriched['customer']).first().sort_index().to_numpy()
    ).max()
)
print('demo npz round-trip (disk vs in-memory, customer-aligned): max |delta| = %.3e' % disk_delta)
assert disk_delta < 1e-6

index-aligned fusion rebuild: max |delta| = 0.000e+00 over 5302 x 1280 features
demo npz round-trip (disk vs in-memory, customer-aligned): max |delta| = 0.000e+00


### Interpretation (read after the tables — null results are findings)

Fill from the tables above, in the F3 report's vocabulary:

- On the **PRIMARY axis** every F4 arm sits at/below chance with CIs that cover it — the
  pre-registered expectation holds; the fused space adds nothing the one-hot tabular matrix
  does not already carry at this scale.
- On the **chronological axis** per-customer embedding blocks (face especially) can separate
  — that is customer-IDENTITY memorization leaking across the time split, the same artifact
  F3 exposed at 0.24 lift, not transferable fraud signal. On both customer-grouped axes the
  identity channel is severed by construction, and the embeddings collapse to chance — the
  honest reading is that appearance/text/profile geometry carries no fraud signal here.
- The `latent-pure` ablation (embeddings only, no tabular base) exists to attribute: if it
  matches `latent-fusion`, the XGB rides the embedding geometry; if it collapses while fusion
  merely ties xgb-client, the tabular base does all the work.

The value delivered is comparability: four cached, label-free, deterministic feature-sets in
the runner, re-runnable against future data through `fraud_pipeline.py --run-arm ...` in
minutes — not a winner at 5.3k transactions.

## Summary

- **Demo embeddings** (`demo-features`): 100 dim_customer profiles -> MiniLM-L6 384-d unit
  vectors (same backbone as the chosen text arm), per-customer constant, label-free,
  `data/demo_embeddings.npz` committed with corpus digest + scheme provenance.
- **Fusion** (`latent-fusion`): stateless concat of L2-normalized face(512)+text(384)+demo(384)
  blocks = 1280-d joint space on top of base+client features. Chosen over fitted
  projection/alignment for a leakage argument (build_features runs pre-split) and a data-size
  argument (100 customers).
- **Determinism**: two cache-bypass fusion builds aligned on ROW IDENTITY — max |delta| = 0
  (bit-identical); demo npz round-trips from disk bit-identically.
- All three axes evaluated through the runner protocol; verdicts printed against chance with
  test positives next to every table.